In [ ]:
pip install transformers datasets torch torchaudio soundfile librosa

In [ ]:
import torch
from PIL import Image
import requests
from transformers import CLIPProcessor, CLIPModel

# --- 1. Load Pretrained CLIP ---
print("Loading CLIP (Image-Text) Model...")
model_id = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_id)
processor = CLIPProcessor.from_pretrained(model_id)

# --- 2. Get an Image ---
# We download a picture of cats
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

# --- 3. Define Text Options ---
texts = ["a photo of a remote control", "a photo of a dog", "a photo of a cat"]

# --- 4. Process and Run ---
inputs = processor(text=texts, images=image, return_tensors="pt", padding=True)

with torch.no_grad():
    outputs = model(**inputs)
    logits_per_image = outputs.logits_per_image  # Similarity score
    probs = logits_per_image.softmax(dim=1)      # Convert to probability

# --- 5. Results ---
print("\n--- Image & Text Results ---")
for i, text in enumerate(texts):
    print(f"Text: '{text}' \t Score: {probs[0][i].item():.4f}")

best_idx = probs.argmax().item()
print(f"CLIP thinks this image is: '{texts[best_idx]}'")

In [ ]:
from transformers import ClapModel, ClapProcessor
import torchaudio
import numpy as np
import scipy.io.wavfile as wavfile # Import for saving WAV files

# --- 1. Load Pretrained CLAP ---
print("\n⏳ Loading CLAP (Audio-Text) Model...")
# This model aligns Audio and Text in a shared space
clap_model_id = "laion/clap-htsat-unfused"
clap_model = ClapModel.from_pretrained(clap_model_id)
clap_processor = ClapProcessor.from_pretrained(clap_model_id)

# --- 2. Get a Sound File ---

local_audio_filename = "/content/4911.wav"

# Load audio with torchaudio
waveform, sample_rate = torchaudio.load(local_audio_filename)

# Resample to 48kHz (required by this specific CLAP model)
if sample_rate != 48000:
    resampler = torchaudio.transforms.Resample(sample_rate, 48000)
    waveform = resampler(waveform)
    sample_rate = 48000

# Convert to mono if needed
if waveform.shape[0] > 1:
    waveform = waveform.mean(dim=0, keepdim=True)

# Squeeze to 1D array
audio_data = waveform.squeeze().numpy()

# --- 3. Define Text Options ---
# One correct description, two wrong ones
audio_texts = ["The sound of a dog barking", "Upbeat instrumental music", "A person speaking quietly"]

# --- 4. Process and Run ---
inputs = clap_processor(text=audio_texts, audio=audio_data, return_tensors="pt", sampling_rate=48000, padding=True)

with torch.no_grad():
    outputs = clap_model(**inputs)
    # Calculate similarity (Dot product of audio and text embeddings)
    logits_per_audio = outputs.logits_per_audio
    probs = logits_per_audio.softmax(dim=1)

# --- 5. Results ---
print("\n--- Sound & Text Results ---")
for i, text in enumerate(audio_texts):
    print(f"Text: '{text}' \t Score: {probs[0][i].item():.4f}")

best_audio_idx = probs.argmax().item()
print(f"CLAP thinks this sound is: '{audio_texts[best_audio_idx]}'")

In [ ]:
print("\n--- 🔗 Image & Sound (Bridging) ---")

# Scenario: We have an Image of a 'Cat' and a Sound of a 'Dog'.
# We want to know if they match.

# 1. We define the concept explicitly
concept_text = ["A cat meowing", "A dog barking"]

# 2. ASK CLIP (Image Check): Does the image match 'Cat' or 'Dog'?
image_inputs = processor(text=concept_text, images=image, return_tensors="pt", padding=True)
img_probs = image_inputs.pixel_values

clip_prediction_index = 0

# 3. ASK CLAP (Audio Check): Does the sound match 'Cat' or 'Dog'?

audio_inputs = clap_processor(text=concept_text, audio=audio_data, return_tensors="pt", sampling_rate=48000, padding=True, truncation=True)
with torch.no_grad():
    audio_outputs = clap_model(**audio_inputs)
    audio_probs = audio_outputs.logits_per_audio.softmax(dim=1)

# 4. Compare
best_audio_match = audio_probs.argmax().item()

print(f"Image Concept: {concept_text[clip_prediction_index]}")
print(f"Audio Concept: {concept_text[best_audio_match]}")

if clip_prediction_index == best_audio_match:
    print("MATCH: The image and sound are about the same thing!")
else:
    print("NO MATCH: The image and sound are different concepts.")